# STAGE E/5 — **v3** inference → zip (A/B assertion)

**Add Input (BẮT BUỘC):** **Stage C** (ner + assert) + **Stage D** (kb).
**Settings:** GPU **T4 (1 con)**. **~6′**. Xuất 2 zip để A/B, nộp cái điểm cao hơn.

**output_ner.zip** = assertion HỌC (Head B) + abbrev + linking chặt. **output_ctx.zip** = ConText rule.


In [ ]:
# Clone repo
import os, subprocess
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # 1 GPU: tránh DataParallel chậm
REPO = "https://github.com/Khanhhh239/Medical_Retrieve.git"
D = "/kaggle/working/Medical_Retrieve"
if not os.path.isdir(D):
    subprocess.run(["git","clone","-q",REPO,D], check=True)
else:
    subprocess.run(["git","-C",D,"pull","-q"])
os.chdir(D); print('cwd:', os.getcwd())


In [ ]:
!pip install -q rapidfuzz pyyaml regex accelerate sentencepiece


In [ ]:
# Tìm artifact từ stage trước (đã Add Input)
import glob, os, shutil
def find_input(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    return hits[0] if hits else None
def find_model(name):
    # model dir = thư mục có config.json (TRÁNH khớp nhầm src/ner là code)
    hits = [h for h in glob.glob(f"/kaggle/input/**/{name}/config.json", recursive=True)
            if "/src/" not in h]
    return os.path.dirname(hits[0]) if hits else None


In [ ]:
import glob, os, shutil
NER = find_model('ner')
ASSERT = find_model('assert')
KB = find_input('kb')
if KB:                                   # nạp KB cho linker
    for f in glob.glob(KB + '/*'):
        if os.path.isfile(f): shutil.copy(f, 'data/kb/' + os.path.basename(f))
print('NER:', NER, '| ASSERT:', ASSERT, '| KB:', bool(KB))
assert NER, 'CHUA ADD INPUT Stage C (ner)!'


In [ ]:
# BẢN v3 CHÍNH: assertion HỌC (Head B) + abbrev + linking chặt
_am = f'--assert_model {ASSERT}' if ASSERT else ''
!python scripts/predict.py --pipeline ner --model_dir {NER} {_am} --out_dir output_ner --zip


In [ ]:
# A/B: assertion ConText (rule) — so xem Head B có hơn không
!python scripts/predict.py --pipeline ner --model_dir {NER} --out_dir output_ctx --zip


In [ ]:
# Copy zip ra /kaggle/working
import shutil, os
for z in ["output_ner.zip","output_ctx.zip"]:
    if os.path.exists(z): shutil.copy(z, '/kaggle/working/'+z); print('SAVED', z)
print(">>> Nộp output_ner.zip (v3) truoc; roi output_ctx.zip de so <<<")


**Thử ABSTENTION** (cắt span thừa → nâng cả 3 điểm): thêm `--min_prob 0.75` vào cell v3 chính, chạy lại → zip khác, nộp so. Thử 0.6 / 0.75 / 0.85 tìm điểm cao nhất.
